# Week 2 · Day 4 — CrewAI: Multi-Agent Collaboration, Roles & Task Delegation

**Business task:** review our laptop product catalog, generate customer-
segment insights, and write a stakeholder-ready summary for a
back-to-school marketing campaign.

**Backend:** `gemini/gemini-3.5-flash-lite` via CrewAI's `LLM` class —
chosen specifically because CrewAI's multi-agent, multi-task runs make
far more LLM calls per run than a single agent, and the "flash-lite"
tier's free-tier quota is substantially higher than the full
`gemini-3.6-flash` model's, which is necessary headroom to run both a
sequential AND a hierarchical crew today. (`gemini-2.0-flash-lite`, used
in an earlier draft, was retired by Google on 2026-07-21 in favor of the
3.5 line — check https://ai.google.dev/gemini-api/docs/rate-limits for
current quotas before relying on a specific number.)

**How to run (Google Colab):**
1. Upload `crew_tools.py`, `crew_agents.py`, `crew_tasks.py`,
   `crew_build.py`, and `products.json` into the same Colab session.
2. Use the same free Gemini API key from earlier days.
3. Runtime → Run all. The API key prompt uses `getpass`, so it's never
   saved into the notebook file.

In [1]:
!pip install -q crewai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.9/198.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/4

In [2]:
import os
import getpass

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]  # some CrewAI paths check this name too

print("API key set:", bool(os.environ.get("GEMINI_API_KEY")))

Enter your Gemini API key: ··········
API key set: True


## Task 1 — Multi-Agent Design Thinking

**The task decomposed into 3 non-overlapping roles:**

| Agent | Role | Goal | Backstory (persona) |
|---|---|---|---|
| **Data Analyst** | Fact retrieval only | Extract precise, accurate facts from the catalog — no opinions, no rounding | A meticulous analyst who's seen bad decisions come from "remembered" prices instead of checked ones |
| **Market Strategist** | Interpretation only | Turn raw facts into 2-3 customer-segment recommendations, each backed by a computed value metric | A strategist who always grounds a recommendation in a specific number, not intuition |
| **Report Writer** | Synthesis only | Produce a polished, under-200-word executive summary from the strategist's segments | A communications specialist who reorganizes and polishes, never invents new facts |

Each boundary is enforced two ways at once: the goal/backstory text, AND
tool access (see Task 2) — the Analyst is the only agent that can touch
raw data, the Strategist is the only one that can compute a value metric,
and the Writer has no tools at all, so it structurally cannot introduce
new facts even if it wanted to.

### Why specialized agents might outperform one generalist here — and where that isn't true

Splitting this task helps because each role has a genuinely different
*failure mode* to guard against: the Analyst's job is to never
hallucinate a number, the Strategist's is to never skip a segment, and
the Writer's is to never bury the recommendation in jargon — a single
generalist agent juggling all three concerns in one prompt is more likely
to blend "just report the facts" instructions with "now be persuasive"
instructions and quietly round a number while trying to sound
compelling. This decomposition is **not** worth it for a task this small
if latency and cost matter more than that specific failure-mode
isolation — three sequential LLM calls (or more, with a manager) will
always be slower and pricier than one well-prompted single agent that
manages the fact/opinion boundary itself with a good system prompt, which
is exactly the trade-off examined empirically in Task 5.

## Task 2 — Build Agents & Assign Tools

Implemented in `crew_agents.py` / `crew_tools.py`:

- **Data Analyst** → `list_products` tool only (the sole source of
  ground-truth catalog data).
- **Market Strategist** → `price_value_calculator` tool only (computes
  price-per-GB-RAM to back a segment recommendation with a real number).
- **Report Writer** → **no tools at all** — a deliberate choice: its job
  is pure synthesis of what's already been produced upstream, and giving
  it tool access would just create an opportunity to re-fetch or
  re-compute something inconsistently with what was already reviewed.

This is the "keep tool access role-appropriate" requirement in practice:
no agent has a tool it doesn't structurally need for its own boundary.

In [3]:
from crew_agents import get_llm, build_agents, build_manager
from crew_build import build_sequential_crew, build_hierarchical_crew

llm = get_llm()
analyst, strategist, writer = build_agents(llm)

for agent in (analyst, strategist, writer):
    print(f"--- {agent.role} ---")
    print("goal:", agent.goal)
    print("tools:", [t.name for t in agent.tools])
    print()

--- Product Data Analyst ---
goal: Extract precise, accurate facts and figures from the laptop product catalog to ground all downstream analysis. Never round numbers, never add opinions, never recommend anything.
tools: ['list_products']

--- Market Strategist ---
goal: Translate the analyst's raw catalog facts into 2-3 distinct customer-segment recommendations, each backed by a real computed value metric rather than a gut-feel guess.
tools: ['price_value_calculator']

--- Stakeholder Report Writer ---
goal: Produce a concise, polished, stakeholder-ready executive summary (under 200 words) recommending which laptops to feature in the campaign, based ONLY on the strategist's segment recommendations.
tools: []



## Task 3 — Define Tasks & Process (Sequential)

`crew_tasks.py`'s `build_sequential_tasks()` wires the three `Task`
objects together with `context=[...]` dependencies so each later task
receives the earlier tasks' outputs automatically.

### The format-mismatch fix (documented in `crew_tasks.py`'s docstring)

An early draft of the analyst's `expected_output` just said "list the
products and their prices" — loose enough that the model paraphrased
numbers ("around $1,500" instead of `1499`), which then broke the
strategist's `price_value_calculator` call in Task 2, since dividing a
paraphrased string isn't a valid arithmetic expression. The fix: the
`expected_output` shipped below explicitly forbids rounding or
paraphrasing and requires the exact `price_usd` as a plain integer — a
concrete example of a downstream tool's needs dictating an upstream
task's `expected_output` wording, not just its `description`.

### Colab/Jupyter fix: `crew.kickoff()` -> `await crew.kickoff_async()`

Running `crew.kickoff()` synchronously inside a Colab/Jupyter cell threw:

```
RuntimeError: Agent execution was invoked synchronously from within a running
event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or
`await agent.aexecute_task(...)`) when calling from async code.
```

**Cause:** Colab/Jupyter notebook cells already run inside an active asyncio
event loop (the kernel's own loop). This version of CrewAI's synchronous
`kickoff()` path detects that an event loop is already running and refuses to
start a second, nested one — a plain script (no pre-existing loop) wouldn't
hit this, which is why it only shows up in a notebook environment.

**Fix:** call `crew.kickoff_async()` instead of `crew.kickoff()`, and `await`
it directly in the cell (Colab/Jupyter cells support top-level `await`).
`kickoff_async()` runs the crew in a background thread
(`asyncio.to_thread`), so it no longer collides with the kernel's own running
loop. No other code changes are needed — `sequential_result` /
`hierarchical_result` are still plain `CrewOutput` objects afterward, so
`.raw` and `.token_usage` work exactly the same as before.

In [4]:
sequential_crew = build_sequential_crew(llm)
sequential_result = await sequential_crew.kickoff_async()

print("\n\n=== FINAL SEQUENTIAL OUTPUT ===\n")
print(sequential_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cc9d4dad-6337-43cc-991a-a129a89188f5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Call the list_products tool to get the full laptop catalog. Report EVERY product with its EXACT          │
│  price_usd, category, and specs, copied verbatim from the tool's output. Do NOT round any number, do NOT        │
│  paraphrase any spec, and do NOT add any opinion, comparison, or recommendation — facts only.                   │
│  ID: 8aed74d4-353d-421d-ba01-52dfef5c5277                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Product Data Analyst                                                                                    │
│                                                                                                                 │
│  Task: Call the list_products tool to get the full laptop catalog. Report EVERY product with its EXACT          │
│  price_usd, category, and specs, copied verbatim from the tool's output. Do NOT round any number, do NOT        │
│  paraphrase any spec, and do NOT add any opinion, comparison, or recommendation — facts only.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool list_products executed with result: {
  "ultrabook pro": {
    "price_usd": 1499,
    "category": "laptop",
    "specs": "14-inch, 16GB RAM, 512GB SSD"
  },
  "budgetbook lite": {
    "price_usd": 549,
    "category": "laptop",
    "spe...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: list_products                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stakeholder Report Writer                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Our back-to-school marketing campaign will target three distinct buyer segments to maximize engagement and     │
│  sales.                                                                                                         │
│                                                                                                                 │
│  For budget-conscious students, we recommend the Budgetbook Lite. Offering an economical price-to-memory ratio  │
│  of $68.63 per GB of RAM ($549 for 8GB), this lightweight 14-inch machine delivers dependable everyday          │
│  performance without straining student finances.                                                                │
│                                                                                                                 │
│  For portability-focused professionals, we recommend the Aircase 13. Designed for effortless travel, this       │
│  compact 13-inch laptop balances necessary everyday multitasking capability with a highly portable form         │
│  factor.                                                                                                        │
│                                                                                                                 │
│  For performance-focused power users, we recommend the Ultrabook Pro. Delivering an efficient $93.69 per GB of  │
│  RAM ($1499 for 16GB), this 14-inch powerhouse provides the elevated memory and storage needed for demanding    │
│  professional workflows.                                                                                        │
│                                                                                                                 │
│  **Hero Product Pick:** Ultrabook Pro                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using ONLY the strategist's segment recommendations above (do not introduce any new fact, product, or    │
│  number), write a concise, polished executive summary under 200 words for a back-to-school marketing campaign.  │
│  Open with a one-sentence overview, one short paragraph per segment/recommended laptop, and close with a        │
│  single named 'hero product' pick for the campaign.                                                             │
│  Agent: Stakeholder Report Writer                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



=== FINAL SEQUENTIAL OUTPUT ===

Our back-to-school marketing campaign will target three distinct buyer segments to maximize engagement and sales.

For budget-conscious students, we recommend the Budgetbook Lite. Offering an economical price-to-memory ratio of $68.63 per GB of RAM ($549 for 8GB), this lightweight 14-inch machine delivers dependable everyday performance without straining student finances.

For portability-focused professionals, we recommend the Aircase 13. Designed for effortless travel, this compact 13-inch laptop balances necessary everyday multitasking capability with a highly portable form factor.

For performance-focused power users, we recommend the Ultrabook Pro. Delivering an efficient $93.69 per GB of RAM ($1499 for 16GB), this 14-inch powerhouse provides the elevated memory and storage needed for demanding professional workflows.

**Hero Product Pick:** Ultrabook Pro


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [5]:
print("--- Sequential run token usage ---")
print(sequential_result.token_usage)

--- Sequential run token usage ---
total_tokens=11034 prompt_tokens=9306 cached_prompt_tokens=0 completion_tokens=1728 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=18


## Task 4 — Hierarchical Delegation

In [6]:
hierarchical_crew = build_hierarchical_crew(llm)
hierarchical_result = await hierarchical_crew.kickoff_async()

print("\n\n=== FINAL HIERARCHICAL OUTPUT ===\n")
print(hierarchical_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3b68702e-7db4-4390-8d5c-8db65a61c170                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cc9d4dad-6337-43cc-991a-a129a89188f5                                                                       │
│  Final Output: Our back-to-school marketing campaign will target three distinct buyer segments to maximize      │
│  engagement and sales.                                                                                          │
│                                                                                                                 │
│  For budget-conscious students, we recommend the Budgetbook Lite. Offering an economical price-to-memory ratio  │
│  of $68.63 per GB of RAM ($549 for 8GB), this lightweight 14-inch machine delivers dependable everyday          │
│  performance without straining student finances.                                                                │
│                                                                                                                 │
│  For portability-focused professionals, we recommend the Aircase 13. Designed for effortless travel, this       │
│  compact 13-inch laptop balances necessary everyday multitasking capability with a highly portable form         │
│  factor.                                                                                                        │
│                                                                                                                 │
│  For performance-focused power users, we recommend the Ultrabook Pro. Delivering an efficient $93.69 per GB of  │
│  RAM ($1499 for 16GB), this 14-inch powerhouse provides the elevated memory and storage needed for demanding    │
│  professional workflows.                                                                                        │
│                                                                                                                 │
│  **Hero Product Pick:** Ultrabook Pro                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Call the list_products tool to get the full laptop catalog. Report EVERY product with its EXACT          │
│  price_usd, category, and specs, copied verbatim from the tool's output. Do NOT round any number, do NOT        │
│  paraphrase any spec, and do NOT add any opinion, comparison, or recommendation — facts only.                   │
│  ID: 4f9358cd-40a6-4a87-aeca-50525a382de0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campaign Project Manager                                                                                │
│                                                                                                                 │
│  Task: Call the list_products tool to get the full laptop catalog. Report EVERY product with its EXACT          │
│  price_usd, category, and specs, copied verbatim from the tool's output. Do NOT round any number, do NOT        │
│  paraphrase any spec, and do NOT add any opinion, comparison, or recommendation — facts only.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'coworker': 'Product Data Analyst', 'context': "We need to get the full laptop catalog from the         │
│  list_products tool and report EVERY product with its EXACT price_usd, category, and specs, copied verba...     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#4) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: **Executive Summary: Back-to-School Laptop Campaign**                                                  │
│                                                                                                                 │
│  This campaign will strategically target three distinct student and professional segments with tailored laptop  │
│  offerings to maximize engagement and conversion.                                                               │
│                                                                                                                 │
│  *   **Budget-Conscious Students:** Feature the Budgetbook Lite ($549, 14-inch, 8GB RAM, 256GB SSD) to appeal   │
│  to cost-sensitive buyers with its strong $68.63 price-per-GB-RAM value.                                        │
│  *   **Portability-Focused Professionals:** Feature the Aircase 13 ($999, 13-inch, 8GB RAM, 256GB SSD) to       │
│  capture users prioritizing a compact form factor at a $124.88 price-per-GB-RAM value.                          │
│  *   **Performance-Focused Power Users:** Feature the Workstation Max ($2799, 16-inch, 32GB RAM, 1TB SSD) to    │
│  attract high-end users seeking robust specifications at an $87.47 price-per-GB-RAM value.                      │
│                                                                                                                 │
│  **Hero Product Pick:** Budgetbook Lite                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campaign Project Manager                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Executive Summary: Back-to-School Laptop Campaign**                                                          │
│                                                                                                                 │
│  This back-to-school marketing campaign strategically targets three distinct audience segments with tailored    │
│  laptop offerings to maximize engagement and conversion.                                                        │
│                                                                                                                 │
│  For budget-conscious students, the Budgetbook Lite delivers an economical price of $549 with 8GB RAM and a     │
│  256GB SSD, providing an exceptional price-per-GB-RAM value of $68.63 for everyday needs.                       │
│                                                                                                                 │
│  For portability-focused professionals, the Aircase 13 features a compact 13-inch form factor with 8GB RAM and  │
│  a 256GB SSD at $999, offering a highly portable solution with a price-per-GB-RAM value of $124.88.             │
│                                                                                                                 │
│  For performance-focused power users, the Workstation Max is built for intensive tasks with 32GB RAM and a 1TB  │
│  SSD at $2799, supporting demanding workloads at an efficient price-per-GB-RAM value of $87.47.                 │
│                                                                                                                 │
│  **Hero Product Pick:** Budgetbook Lite                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using ONLY the segment recommendations produced above (do not introduce any new fact, product, or        │
│  number), write a concise, polished executive summary under 200 words for a back-to-school marketing campaign.  │
│  Open with a one-sentence overview, one short paragraph per segment/recommended laptop, and close with a        │
│  single named 'hero product' pick for the campaign.                                                             │
│  Agent: Campaign Project Manager                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



=== FINAL HIERARCHICAL OUTPUT ===

**Executive Summary: Back-to-School Laptop Campaign**

This back-to-school marketing campaign strategically targets three distinct audience segments with tailored laptop offerings to maximize engagement and conversion.

For budget-conscious students, the Budgetbook Lite delivers an economical price of $549 with 8GB RAM and a 256GB SSD, providing an exceptional price-per-GB-RAM value of $68.63 for everyday needs.

For portability-focused professionals, the Aircase 13 features a compact 13-inch form factor with 8GB RAM and a 256GB SSD at $999, offering a highly portable solution with a price-per-GB-RAM value of $124.88.

For performance-focused power users, the Workstation Max is built for intensive tasks with 32GB RAM and a 1TB SSD at $2799, supporting demanding workloads at an efficient price-per-GB-RAM value of $87.47.

**Hero Product Pick:** Budgetbook Lite


In [7]:
print("--- Hierarchical run token usage ---")
print(hierarchical_result.token_usage)

--- Hierarchical run token usage ---
total_tokens=75360 prompt_tokens=61448 cached_prompt_tokens=0 completion_tokens=13912 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=76


### Sequential vs. hierarchical — pros, cons, when to use each

| | Sequential | Hierarchical |
|---|---|---|
| **Pros** | Predictable, fixed order; cheapest (no manager overhead); easiest to debug — you know exactly which agent ran when | Manager can re-route or reject sub-agent output before it propagates; better suited to open-ended tasks where the right agent/order isn't known in advance |
| **Cons** | No review step — a bad Task 1 output silently flows into Task 2 and Task 3 with nothing catching it | Extra LLM calls for the manager's own delegation + review reasoning → higher token usage and latency for the same underlying work |
| **When to use** | The task's structure and agent-to-task mapping is already known and fixed (like this one) | The task is more open-ended, sub-agent output quality is more variable, or you specifically want an automated review/rejection step before results propagate |

## Task 5 — Evaluation & Cost Awareness

### Token usage comparison

In [8]:
def summarize_usage(label, token_usage):
    print(f"{label}:")
    print(f"  prompt_tokens:     {token_usage.prompt_tokens}")
    print(f"  completion_tokens: {token_usage.completion_tokens}")
    print(f"  total_tokens:      {token_usage.total_tokens}")
    print(f"  successful_requests: {token_usage.successful_requests}")
    print()

summarize_usage("Sequential crew", sequential_result.token_usage)
summarize_usage("Hierarchical crew", hierarchical_result.token_usage)

Sequential crew:
  prompt_tokens:     9306
  completion_tokens: 1728
  total_tokens:      11034
  successful_requests: 18

Hierarchical crew:
  prompt_tokens:     61448
  completion_tokens: 13912
  total_tokens:      75360
  successful_requests: 76



╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3b68702e-7db4-4390-8d5c-8db65a61c170                                                                       │
│  Final Output: **Executive Summary: Back-to-School Laptop Campaign**                                            │
│                                                                                                                 │
│  This back-to-school marketing campaign strategically targets three distinct audience segments with tailored    │
│  laptop offerings to maximize engagement and conversion.                                                        │
│                                                                                                                 │
│  For budget-conscious students, the Budgetbook Lite delivers an economical price of $549 with 8GB RAM and a     │
│  256GB SSD, providing an exceptional price-per-GB-RAM value of $68.63 for everyday needs.                       │
│                                                                                                                 │
│  For portability-focused professionals, the Aircase 13 features a compact 13-inch form factor with 8GB RAM and  │
│  a 256GB SSD at $999, offering a highly portable solution with a price-per-GB-RAM value of $124.88.             │
│                                                                                                                 │
│  For performance-focused power users, the Workstation Max is built for intensive tasks with 32GB RAM and a 1TB  │
│  SSD at $2799, supporting demanding workloads at an efficient price-per-GB-RAM value of $87.47.                 │
│                                                                                                                 │
│  **Hero Product Pick:** Budgetbook Lite                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Comparison vs. Day 3's single-agent LangGraph solution

Day 3's LangGraph agent solved a related-but-simpler task (compare two
named laptops for one client) with 2 LLM calls per pass (draft +
critique) and no manager overhead — genuinely captured Day 3 output:

> *"For a budget-conscious client, the BudgetBook Lite is the better
> choice. At \$549, it is significantly less expensive than the
> UltraBook Pro, which costs \$1,499..."*

That's a narrower task than today's (comparing 2 named products vs.
segmenting the entire catalog into 3 personas), so a direct token-for-
token comparison isn't apples-to-apples — but it's illustrative: Day 3's
single-purpose LangGraph agent needed only 2 calls to fully solve its
(narrower) task, while today's sequential crew needs at least 3 (one per
agent) and the hierarchical version needs more still for manager
overhead, for a broader task.

### 3 success criteria, manually scored across 3 runs

| Criterion | What it checks |
|---|---|
| **Factual grounding** | Every price/spec cited in the final output exactly matches `products.json` — no invented numbers |
| **Completeness** | All 4 catalog products are considered, all 2-3 segments are covered, and a single hero product is named |
| **Tone** | Reads as a polished, concise stakeholder summary — not an internal analyst note or a bulleted data dump |

Score each run 1 (fails), 2 (partial), or 3 (fully meets) per criterion
after reading its actual output above:

| Run | Factual grounding | Completeness | Tone |
|---|---|---|---|
| Sequential crew | 3 — Budgetbook Lite ($549/8GB) and Ultrabook Pro ($1499/16GB) prices/specs and the $68.63 and $93.69 price-per-GB-RAM figures all match `products.json` exactly | 2 — 3 segments covered and one hero product named, but Workstation Max never appears in the final output, so not all 4 catalog products were considered | 3 — reads as a clean, polished stakeholder summary ending in one clearly labeled hero pick |
| Hierarchical crew | 3 — Budgetbook Lite, Aircase 13, and Workstation Max prices/specs and their $68.63 / $124.88 / $87.47 price-per-GB-RAM figures all match `products.json` exactly | 2 — 3 segments covered and one hero product named, but Ultrabook Pro never appears in the final output, so (like the sequential run) not all 4 products were considered | 3 — reads as a clean executive summary, comparable in polish to the sequential run |
| Day 3 LangGraph (single-agent) | 3 — every figure matches the catalog exactly | 2 — solved its own narrower 2-product task fully, but wasn't designed to segment the whole catalog | 3 — reads as a clean, direct recommendation |

**Observation:** both crew runs independently dropped one product from the final narrative (Sequential omitted Workstation Max; Hierarchical omitted Ultrabook Pro) even though the Analyst's raw catalog facts always contained all 4 — worth noting since neither process, on its own, guarantees every product survives into the final summary. The two runs also picked **different hero products** (Ultrabook Pro vs. Budgetbook Lite) from the same catalog and the same underlying instructions, which is itself a useful reliability data point for Task 4's comparison: the manager-mediated hierarchical run isn't necessarily more consistent, just differently shaped.

### Was the multi-agent crew worth it here?

For this specific task, splitting fact-retrieval, strategy, and writing
into three agents is a defensible but not obviously necessary choice: the
task is bounded and well-understood enough that a single, carefully
prompted agent with both tools (catalog lookup + calculator) and a clear
"facts first, then reasoning, then a polished summary" system prompt
could very plausibly hit the same 3 success criteria in one or two LLM
calls, at a fraction of the token cost of even the sequential crew and
without the hierarchical version's manager overhead at all. The multi-
agent structure earns its cost more clearly if this task grows — more
data sources, more customer segments, or a real need for an automated
review gate before a stakeholder ever sees the output — none of which
this specific instance of the task actually required.